In [1]:
!pip install -q langchain langchain-community langchain-chroma chromadb sentence-transformers pypdf


In [2]:
!pip install -q langchain-google-genai

In [3]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_30478/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
!pip install -q -U langchain


In [5]:
file_path= "/content/Manish resume -june.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(f"successflly fetched your data!")
print(f"Total page found: {len(documents)}")

print("\n--- First page Preview---")
print(documents[0].page_content[:500])

successflly fetched your data!
Total page found: 2

--- First page Preview---
MANISH KUMAR
 manishkushwahakr@gmail.com | +91 6204587356 | Noida, India | LinkedIn | GitHub
Professional Summary
Results-driven B.Tech Computer Science student specializing in Data Science and AI/ML Development, with hands-on experience
building machine learning models and AI-powered applications. Proficient in Python, scikit-learn, NLP, and Generative AI tools
(Claude API, OpenAI API). Demonstrated ability to develop scalable end-to-end ML pipelines and deploy production-ready
solutions. Seeki


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20
)

# 2. Split your resume documents
chunks = text_splitter.split_documents(documents)

# 3. Print the confirmation
print(f"Split document into {len(chunks)} smaller chunks.")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk.page_content}")

Split document into 50 smaller chunks.
Chunk 0: MANISH KUMAR
 manishkushwahakr@gmail.com | +91 6204587356 | Noida, India | LinkedIn | GitHub
Professional Summary
Chunk 1: Results-driven B.Tech Computer Science student specializing in Data Science and AI/ML Development, with hands-on experience
Chunk 2: building machine learning models and AI-powered applications. Proficient in Python, scikit-learn, NLP, and Generative AI tools
Chunk 3: (Claude API, OpenAI API). Demonstrated ability to develop scalable end-to-end ML pipelines and deploy production-ready
Chunk 4: solutions. Seeking an entry-level AI/ML role to contribute technical expertise in machine learning, natural language processing,
Chunk 5: and data-driven product development.
Technical Skills
Programming & Data Science:
Chunk 6: Python, SQL (MySQL, PostgreSQL), Data Analysis, Statistics, Machine Learning, Natural
Language Processing (NLP), Scikit-learn, Feature Engineering
Chunk 7: AI/ML & GenAI Tools:
Claude API, OpenAI API (GP

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(chunks, embedding_model)
print("Vector database successfully initialized and populated!")

/tmp/ipykernel_30478/3301802016.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database successfully initialized and populated!


In [8]:
from re import search
retriever = vector_store.as_retriever(search_kwargs={"k":2})
query = "The company provides  CEI Internship?"
retrieved_docs = retriever.invoke(query)
print(f"Query: {query}\n")
print("Top matched chunks found in DB:")
for doc in retrieved_docs:
  print(f"-{doc.page_content}")

Query: The company provides  CEI Internship?

Top matched chunks found in DB:
- Selected as a Celebal Excellence Intern (CEI) for Data Science at Celebal Technologies, a structured industry-focused
-presentation skills.
Professional Experience
Data Science Intern (CEI — Celebal Excellence Intern)
 Jun 2026 – Present
Celebal Technologies, Remote


In [9]:
# 1. Setup your API Key
import os
from google.colab import userdata

# Using the exact name you configured in your Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLEAPI')

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Universal modern LangChain helpers that work across all layout versions:
from langchain.chains import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 2. Define the system instructions (Prompt Engineering)
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know.\n\n"
    "Context:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# 4. Glue the components together into a Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. Run it!
response = rag_chain.invoke({
    "input": "What role or internship did the candidate do at Celebal Technologies?"
})
print("Answer:", response["answer"])

ModuleNotFoundError: No module named 'langchain.chains'